# Network propagation development

The notebook currently covers how results from a AnnData/MuData object can be added to a cpr_graph to setup network-based inference. The general strategy is to:
1. pull out a pd.DataFrame containing feature-level measures of interest along with feature metadata.
2. These are then mapped on the species ids in an sbml_dfs model by shared on ontology, disambiguated (to handle mapping of multiple features to the same s_id), and s_id-indexed results are embedded in the sbml_dfs as a table in species_data
3. attributes of interrest are then passed from the sbml_dfs model into the graph.

This example uses real MuData results but only a small sbml_dfs object which has uniprot but not ENSG identifiers. This makes things easy to work with but a genome-scale graph will need to be used for a real analysis.

Reflecting on the current functionality,

(1) is not too hard but the interface can probably be cleaned up as we should have a function which applies 1-3 in a single call.
(2) is in pretty good shape following a LOT of new functionality being added to napistu-py for handling many-to-one mappings and wide/nested formats for identifiers.
(3) will need some better functionality since the reaction_attrs syntax is pretty cryptic but the core functionality is all there.

Next, steps will be develop basic PPR functionality.

In [1]:
import os

import mudata as md
import pandas as pd

from napistu import utils as napistu_utils
from napistu.network import net_create
from napistu.network import net_propagation
from napistu.network import net_utils
from napistu.gcs import downloads
from napistu.matching import mount
from napistu.scverse.loading import prepare_anndata_results_df
from napistu.scverse.loading import prepare_mudata_results_df
from napistu.matching.constants import BIND_DICT_OF_WIDE_RESULTS_STRATEGIES_LIST

# setup logging
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# paths
PROJECT_DIR =  os.path.expanduser("~/Desktop/DATA/Forny2023")
SUPPLEMENTAL_DATA_DIR = os.path.join(PROJECT_DIR, "input")
CACHE_DIR = os.path.join(PROJECT_DIR, "cache")
NAPISTU_DATA_DIR = os.path.expanduser("~/Desktop/DATA/napistu_data")

# Define the path to save hyperparameter scan results
MOFA_PARAM_SCAN_MODELS_PATH = os.path.join(CACHE_DIR, "mofa_param_scan_h5mu")
# Final results 
OPTIMAL_MODEL_H5MU_PATH = os.path.join(CACHE_DIR, "mofa_optimal_model.h5mu")

In [2]:
sbml_dfs_path = downloads.load_public_napistu_asset(
    asset = "human_consensus",
    data_dir = NAPISTU_DATA_DIR,
    subasset = "sbml_dfs"
)

napistu_graph_path  = downloads.load_public_napistu_asset(
    asset = "human_consensus",
    data_dir = NAPISTU_DATA_DIR,
    subasset = "regulatory_graph"
)

identifiers_path = downloads.load_public_napistu_asset(
    asset = "human_consensus",
    data_dir = NAPISTU_DATA_DIR,
    subasset = "identifiers"
)


In [3]:
# ~2 min load
sbml_dfs = napistu_utils.load_pickle(sbml_dfs_path)
napistu_graph = napistu_utils.load_pickle(napistu_graph_path)
species_identifiers = pd.read_csv(identifiers_path, delimiter = "\t")

In [4]:
#net_utils.validate_assets(
#    sbml_dfs = sbml_dfs,
#    cpr_graph = cpr_graph,
#    # TODO - it should really possible for this to be optional
#    precomputed_distances = None,
#    identifiers = species_identifiers
#    )

In [ ]:
# lets load the Forny results so we trying adding a few different types of tables to the sbml_dfs
mdata = md.read_h5mu(OPTIMAL_MODEL_H5MU_PATH)

## Adding genome-scale datasets

To use an 'omic dataset in Napistu, we want to:
1. mount the dataset on the pathway `sbml_dfs`. This entails:
    - matching systematic identifiers between the dataset and pathway to connect 'omic features to Napistu `species`.
    - resolve many-to-1 mappings (e.g., where 2+ features match the same species).
    - create a table with unique species ids as the index with variable from the dataset.
    - add this to the `species_data` attriute of the `sbml_dfs`. Multiple tables and/or datasets can be added to `species_data`.
2. pass variables from one or more `species_data` tables to a `napistu_graph`'s vertices with `net_create._add_graph_species_attribute`. Variables can be transformed (e.g., to make them non-negative for personalized pagerank) at this point (or this could be done before step (1)).
3. use these verterx attributes for downstream analysis (e.g., using it in the reset_proportional_to parameters of PPR).

Step (1) needs to be adapted depending on how datasets are organized. The currently, supported inputs are:
- `pd.DataFrame` objects which including 1+ systematic identifiers
- `anndata.AnnData` objects where the `var` table provided identifiers, and feature-level summaries come from either the `var`, `varm` or `X` tables.
- `mudata.MuData` objects containing multiple `AnnData` objects where `var` and `varm` attributes can be defined across multiple datasets.

We'll provide an examples using each of these inputs

### Loading results from a pd.DataFrame

In [6]:
SUPPLEMENTAL_DATA_DIR = os.path.join(PROJECT_DIR, "input")
VZ_LMM_RESULTS = {
    "transcriptomics": "diff_exp_lmm_rnaseq_pathwayact_all_annotout.txt",
    "proteomics": "diff_exp_lmm_prot_pathwayact_all_annotout.txt"
}

sideloaded_data_path = {x : os.path.join(SUPPLEMENTAL_DATA_DIR, y) for x, y in VZ_LMM_RESULTS.items()}

assert all([os.path.isfile(x) for x in sideloaded_data_path.values()])

sideloaded_data = {
    x : pd.read_csv(y, delimiter= "\t") for x, y in sideloaded_data_path.items()
}

In [ ]:
for k in sideloaded_data.keys():
    x = sideloaded_data[k][["ensembl", "chi_sq", "pval", "fdr"]]

    mount.bind_wide_results(
        sbml_dfs,
        x,
        f"{k}_loose_data",
        # map columns to Napistu's controlled vocabulary (constants.ONTOLOGIES)
        ontologies = {"ensembl" : "ensembl_gene"},
        species_identifiers = species_identifiers,
        dogmatic = False,
        verbose = True
    )

## Loading Results from an AnnData object

Since the Forny dataset is a multiomics experiment many of the variablges we are interested in will hold a common interpretation across all modalities. For example, the effect size of a term in a regression holds a common meaning as do the loadings from a multi-omic factor analysis (MOFA) decomposition.

But, many datasets will just be a single modality, and even for multiomic datasets we may be interested in exploring the biology of datamodality-specific attributes. An example in this study is the data-modality specific principal component loadings. Since PCA was performed separately on each data modality the principal components will likely be relatively uncorrelated hence it doesn't make much sense to treat the loadings of PCX to one another across modalities. This is definitely the case for this dataset - PC1 of the proteomics data largely reflects a chromatography-driven technical batch effect which is not seen in the transcriptomics data. To more directly explore this proteomics batch effect we can pull PC1 out of its `AnnData` table.

In [8]:
MUDATA_ONTOLOGIES = {
    "transcriptomics" :
        {"ontologies" : ["ensembl_gene"],
         "index_which_ontology" : "ensembl_gene"},
    "proteomics" :
        {"ontologies" : ["uniprot"],
         "index_which_ontology" : "uniprot"}
}

In [ ]:

# TO DO - switch this for PCA loadings. Looks like only the PCs were stored in their AnnDatas

anndata_results_df = prepare_anndata_results_df(
    mdata["proteomics"],
    table_type = "layers",
    index_which_ontology = "uniprot",
    results_attrs = ["MMA001", "MMA004", "MMA005"]
)

mount.bind_wide_results(
    sbml_dfs,
    anndata_results_df,
    "proteomics_var_level_results",
    species_identifiers = species_identifiers,
    ontologies = "uniprot"
)

In [ ]:
# we can look at the species data created thus far
for k, v in sbml_dfs.species_data.items():
    print(k)
    display(napistu_utils.style_df(v.head(5)))


### Loading Results from a MuData object

MuData is a data structure for organizing multiple AnnData objects which can be maninpulated with AnnData-level operations but the combined dataset also has its own attributes which pertain to all data modalities. Napistu provides convenience functions for pulling multi-omic attributes out a MuData object and these can either be stored as a separate attribute for each modality or as a single summary defined over all modalities. The latter workflow may be helpful when combining modalities with non-overlapping ontologies - for example, proteins and metabolites. While, keeping modalities separate may be preferred if multiple modalities would map to the same nodes. For example, working with transcriptomics and proteomics, like in the Forny study, we may want to separately map want to use separate vertex attributes for each modality. This may not be necessary if we are working in "dogmatic" mode where genes, transcripts, and proteins are generally represented as separate nodes, but in non-dogmatic mode these entries are treated equivalently. But, here the network that we are working with was created in non-dogmatic mode ([link](https://github.com/napistu/napistu/blob/26402a440be9d9cb901c1edf371ef0a7e5475e55/dev/create_human_consensus.qmd#L215)).

In [ ]:
split_results_tables = prepare_mudata_results_df(
    mdata,
    mudata_ontologies=MUDATA_ONTOLOGIES,
    table_type="varm",
    table_name="LFs", # this would be autodetected
    results_attrs=["LF1", "LF2", "LF3", "LF4", "LF5"],
    table_colnames=[f"LF{i}" for i in range(1, mdata.varm["LFs"].shape[1] + 1)]
)

for k, v in split_results_tables.items():
    print(k)
    display(napistu_utils.style_df(v.head(5)))


Now, we can can decide how we want to mount these objects on an `SBML_dfs` object. We could either:
- add each modality's results as a separate key-value pair in the species_data attribute
- add them as the same attribute but change the attribute's names to distinguish modalities
- merge them into a single table using the same attribute name for all modalities. This may result in merging of multiple modalities results if they map to the same species.

To handle these different workflows, we can use the `bind_dict_of_wide_results()` function.

In [ ]:
for strategy in BIND_DICT_OF_WIDE_RESULTS_STRATEGIES_LIST:

    mount.bind_dict_of_wide_results(
        sbml_dfs,
        split_results_tables,
        f"{strategy}_results",
        strategy = strategy,
        species_identifiers = species_identifiers,
        # ontologies were already renamed to the controlled vocabulary in prepare_mudata_results_df()
        ontologies = None,
        # ignored because species_identifiers is provided
        dogmatic = False,
        # for clarity; default is True
        inplace = True,
        verbose = False
    )

In [ ]:
results = sbml_dfs.species_data["concatenate_results"]
print(f"Concatenated results; shape: {results.shape}")
display(napistu_utils.style_df(results.head(5)))

results = sbml_dfs.species_data["stagger_results"]
print(f"Staggered results; shape: {results.shape}")
display(napistu_utils.style_df(results.head(5)))

print("Separated results")
for k in split_results_tables.keys():
    species_data_name = f"multiple_keys_results_{k}"
    results = sbml_dfs.species_data[species_data_name]
    print(f"{species_data_name}; shape: {results.shape}")
    display(napistu_utils.style_df(results.head(5)))


In [ ]:
# now we can add .var attributes from the mdata

VAR_VARS = ["tstat_MMA_urine", "qval_MMA_urine", "tstat_OHCblPlus", "qval_OHCblPlus", "tstat_responsive_to_acute_treatment", "qval_responsive_to_acute_treatment", "tstat_date_freezing", "qval_date_freezing"]

split_results_tables = prepare_mudata_results_df(
    mdata,
    mudata_ontologies=MUDATA_ONTOLOGIES,
    table_type="var",
    results_attrs=VAR_VARS
)

In [ ]:
mount.bind_dict_of_wide_results(
    sbml_dfs,
    split_results_tables,
    "var_level_results",
    strategy = "stagger",
    species_identifiers = species_identifiers,
    verbose = False
)

Here, is the final rundown of species_data tables we've added to the `sbml_dfs`:

In [ ]:
for k in sbml_dfs.species_data.keys():
    logger.info(f"{k}: {sbml_dfs.species_data[k].columns.tolist()}")

## Adding Attributes to a Graph

Now, we can pass attributes of interest from species_data tables to the Napistu graph object. For convenience, species attributes are added after a graph's creation and we already loaded a regulatory graph derived from our `sbml_dfs` above.

Attributes are passed using a dictonary which encodes the species_data table, variable name, a transformation, and the vertex attribute name.

In [78]:
from napistu.network import net_create
from napistu.network import data_handling
import numpy as np

CUSTOM_TRANSFORMATIONS = {
    # take the absolute value
    "abs" : lambda x: abs(x),
    # -log10[pvalue]
    "nlog10" : lambda x: -np.log10(x),
    # threshold based on loose FDR threshold and then transform
    "hard_thresholded_nlog10" : lambda x: -np.log10(x) if x < 0.2 else 0,
    "square" : lambda x: x**2
}

GRAPH_ATTRS = {
    "species": {
        "proteomics_chi_sq": {
            "table": "proteomics_loose_data",
            "variable": "chi_sq",
            "trans": "identity",
        },
        "transcriptomics_chi_sq": {
            "table": "transcriptomics_loose_data",
            "variable": "chi_sq",
            "trans": "identity",
        },
        "proteomics_pvalue" : {
            "table": "proteomics_loose_data",
            "variable": "pval",
            "trans": "nlog10",
        },
        "transcriptomics_pvalue" : {
            "table": "transcriptomics_loose_data",
            "variable": "pval",
            "trans": "nlog10",
        },
    },
    "reactions" : {
        "string_wt" : {
            "table" : "string",
            "variable" : "combined_score",
            "trans" : "string_inv"
        }
    }
}

# note that this could have been combined with the LOOSE_GRAPH_ATTRS but we're keeping them separate for clarity
ADD_GRAPH_ATTRS_SPEC = {
    "species": {
        "proteomics_fdr": {
            "table": "proteomics_loose_data",
            "variable": "fdr",
            "trans": "hard_thresholded_nlog10",
        },
        "transcriptomics_fdr": {
            "table": "transcriptomics_loose_data",
            "variable": "fdr",
            "trans": "hard_thresholded_nlog10",
        }
    }


In [ ]:
# let's create a new graph so we can invert the edges so information flows from targets to regulators

napistu_graph = net_create.process_cpr_graph(
    sbml_dfs,
    reaction_graph_attrs = GRAPH_ATTRS,
    directed = True,
    edge_reversed = True,
    graph_type = "regulatory",
    weighting_strategy = "mixed",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

In [ ]:
# we can add vertex attributes after a graph's creation using this function
# this is handy when we are pulling attributes from a bunch of tables and need to use different transformations
napistu_graph = data_handling._add_graph_species_attribute(
    napistu_graph,
    sbml_dfs,
    species_graph_attrs = ADD_GRAPH_ATTRS_SPEC,
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

In [ ]:
sbml_dfs.species_data.keys()
sbml_dfs.species_data["proteomics_var_level_results"]

In [ ]:
# add attributes an AnnData-level table
data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "MMA",
    table_name = "proteomics_var_level_results",
    transformation = "square",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

# add results from the MuData-level mvar table
data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "LF",
    table_name = "stagger_results",
    transformation = "square",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

# add results from the MuData-level var table - regression results
data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "qval",
    table_name = "var_level_results",
    transformation = "hard_thresholded_nlog10",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

data_handling.add_results_table_to_graph(
    napistu_graph,
    sbml_dfs,
    attribute_names = "tstat",
    table_name = "var_level_results",
    transformation = "abs",
    custom_transformations = CUSTOM_TRANSFORMATIONS
)

## Network Propagation

Here we'll implement a workflow for applying network propagation to a cpr_graph's vertex attributes.

In [47]:
from napistu.network import net_propagation


In [ ]:
annotated_vertices = napistu_graph.get_vertex_dataframe()

# find valid attributes - numeric + 1+ non-zero values
invalid_attributes = [x for x in annotated_vertices.columns if annotated_vertices[x].dtype not in ["float64", "int64"] or annotated_vertices[x].nunique() == 1]
invalid_attributes


In [ ]:
netprop_results = dict()

valid_attributes = [x for x in annotated_vertices.columns if x not in invalid_attributes]
for attribute in valid_attributes:

    netprop_results[attribute] = net_propagation.personalized_pagerank_by_attribute(
        napistu_graph,
        attribute
    )

napistu_utils.style_df(netprop_results[list(netprop_results.keys())[0]].head())

In [ ]:
reorganized_results = dict()
for k, v in netprop_results.items():
    v["attribute"] = k
    v = v.drop(columns = [k])
    reorganized_results[k] = v

reorganized_results = pd.concat(list(reorganized_results.values()))